# Predicting NBA Game Totals with Leak-Free Point-in-Time Features

Most NBA totals models quietly cheat: they use stats that include the game being predicted. This notebook uses **point-in-time features** — every number is computed only from games played *before* the game in question.

**Data:** the free sample dataset attached to this notebook — 79 games from the 2026-27 NBA season, 56 features per game (season-to-date team stats blended with prior-season averages, plus last-10-games form), target = `point_total`.

**Full dataset (4,372 games, 2022-23 to 2026-27, with data dictionary + example notebook):** https://kmdspark5.gumroad.com/l/pisdro — launch discount code **LAUNCH25** (25% off until Nov 30).

In [ ]:
import glob, pandas as pd, numpy as np

path = glob.glob('/kaggle/input/**/nba_pit_features_SAMPLE_2026-27.csv', recursive=True)[0]
df = pd.read_csv(path).sort_values('date').reset_index(drop=True)
print(df.shape)
df[['date','home_team','away_team','home_score','away_score','point_total']].head()

In [ ]:
meta = ['game_id','date','season','home_team','away_team','home_score','away_score']
feats = [c for c in df.columns if c not in meta + ['point_total']]
print(len(feats), 'features')
# How do single features relate to the total?
corr = df[feats + ['point_total']].corr()['point_total'].drop('point_total')
corr.reindex(corr.abs().sort_values(ascending=False).index).head(10)

## A quick honest model
Time-based split inside the sample: train on the earlier games, test on the later ones. With only 79 games this is a demo, not a benchmark — treat the numbers as a sanity check.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from math import sqrt

cut = int(len(df) * 0.7)
tr, te = df.iloc[:cut], df.iloc[cut:]
m = GradientBoostingRegressor(random_state=42)
m.fit(tr[feats], tr['point_total'])
pred = m.predict(te[feats])
err = np.abs(pred - te['point_total'])
print(f"test games: {len(te)}")
print(f"model  MAE {mean_absolute_error(te['point_total'], pred):.1f}  RMSE {sqrt(mean_squared_error(te['point_total'], pred)):.1f}  within +/-10: {(err <= 10).mean()*100:.0f}%")
base = np.repeat(tr['point_total'].mean(), len(te))
print(f"naive (train mean) MAE {mean_absolute_error(te['point_total'], base):.1f}")

## The real benchmark (full dataset)
Trained on 2022-23..2024-25 (3,218 games) and tested on the full 2025-26 season (1,075 games), a default GradientBoosting model on these features scores **RMSE 19.56 / MAE 15.56 / 39.2% within ±10 points**. Game totals are intrinsically noisy — distrust any leak-free claim far better than this.

**Get the full dataset:** https://kmdspark5.gumroad.com/l/pisdro (25% off with code **LAUNCH25** until Nov 30)
**Companion product:** the 3-season walk-forward backtest of this model vs closing lines (3,216 games): https://kmdspark5.gumroad.com/l/vkoul